# Nudge Problem: Second-shop test (cosmetics)

**Question:** the Transformer was trained only on the multi-category electronics store. If we put it in a *different* shop, the REES46 cosmetics shop, does it still rank shoppers well? **No retraining**: the same weights in `app/model.pt`, the same logistic regression weights, the same rules. Follows **Week 3 s22** (judge on data the model never saw) and **Week 8 s6** (does the model carry over, or did it learn one shop?).

1. Build visits from the cosmetics clicks with the exact same rules (`scripts/build_cosmetics_visits.py`, same `build_visits`, same 1-in-4 shoppers, Oct and Nov 2019).
2. Score every visit with: random, the industry rule, the logistic regression (3 seeds averaged), the Transformer (3 seeds averaged).
3. Ranking quality: AUC, PR-AUC, top-10% come-back rate, with 95% ranges from resampling shoppers (200 draws).
4. Money at the middle generosity setting (`cutoffs(0.5)`), against the industry rule.
5. Calibration: does a score of 0.2 still mean about 20%, or is the new shop shifted?

**Input:** `data/raw/cosmetics/visits_25pct.parquet` and `build_stats.json` (not in git), `app/model.pt`, `results/tuned/*.pt`.
**Output:** `results/second_shop_quality.csv`, `second_shop_money.csv`, `second_shop_calibration.csv`, `second_shop_summary.json`.

**Owner:** TBD (Comparison role)

## 0. Imports and settings
Every cosmetics visit is scored (no train/valid/test split: no model was trained on this shop, so all of it is test).

In [1]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # repo root, so we can import nudge

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score, average_precision_score
from IPython.display import display

from nudge.contract import VOCAB
from nudge.train import SEED, BLOCK_SIZE, VOCAB_SIZE, set_seed, visits_to_ids, word_counts, make_loader, predict_proba
from nudge.models import LogisticRegression
from nudge.transformer import VisitTransformer
from nudge.evaluate import money, bootstrap, industry_rule_score, top_share_rate
from nudge.decision import cutoffs

set_seed(SEED)
torch.set_num_threads(4)
DATA = pathlib.Path("../data/raw/cosmetics")
OUT = pathlib.Path("../results")
TUNED = OUT / "tuned"
BOOT_N = 200


## 1. Load the cosmetics visits and look at the shift
Cosmetics sell mostly small items. The price bands (cheap under €100, mid €100 to €250, expensive from €250) were cut for electronics, so almost every cosmetics click is the same word type. That is a real distribution shift.

In [2]:
visits = pd.read_parquet(DATA / "visits_25pct.parquet")
stats = json.loads((DATA / "build_stats.json").read_text())
y = visits["label"].to_numpy()
sampled = stats["events_sampled_users_by_type"]
dropped = sampled["remove_from_cart"]
print(f"{len(visits):,} abandoned visits from {stats['shoppers']:,} shoppers | {y.mean():.1%} come back and buy within 7 days")
print(f"cart abandoners: {(visits.cart_value > 0).sum():,} ({(visits.cart_value > 0).mean():.1%}) | returning: {(visits.tokens.str[0] == 'returning').mean():.1%}")
print(f"clicks of sampled shoppers by type: {sampled}")
print(f"remove_from_cart dropped (not in our vocabulary): {dropped:,} of {sum(sampled.values()):,} events ({dropped / sum(sampled.values()):.1%})")
shift = pd.DataFrame({"cosmetics": stats["price_band_share_cosmetics"], "electronics (main)": stats["price_band_share_main"]}).loc[["cheap", "mid", "expensive"]]
display(shift.map(lambda v: f"{v:.1%}"))
print(f"median price: cosmetics {stats['median_price_cosmetics']:.2f} vs electronics {stats['median_price_main']:.2f}")
words = pd.Series([w for t in visits["tokens"] for w in t]).value_counts(normalize=True)
print("share of all words that are *_cheap_*:", f"{words[[w for w in words.index if '_cheap_' in w]].sum():.1%}")

182,980 abandoned visits from 178,442 shoppers | 7.9% come back and buy within 7 days
cart abandoners: 42,554 (23.3%) | returning: 40.4%
clicks of sampled shoppers by type: {'cart': 638905, 'purchase': 144653, 'remove_from_cart': 423878, 'view': 984617}
remove_from_cart dropped (not in our vocabulary): 423,878 of 2,192,053 events (19.3%)


,cosmetics,electronics (main)
cheap,99.0%,34.0%
mid,0.9%,30.1%
expensive,0.1%,35.9%


median price: cosmetics 4.13 vs electronics 165.16
share of all words that are *_cheap_*: 67.6%


## 2. Score every visit with every model
The Transformer: the 3 seeds in `app/model.pt`, batched, with the same truncation (`visits_to_ids`), averaged as in `nudge.contract.predict`. The logistic regression: the 3 seeds of the final setting in `results/tuned/` (tag `aa1b0357`, from `results/evaluation_final.json`), averaged, on word counts. Random: a uniform draw (a coin flip, no information).

In [3]:
ids = visits_to_ids(visits["tokens"])
loader = make_loader(ids, y, batch_size=4096)

bundle = torch.load(OUT.parent / "app" / "model.pt", map_location="cpu", weights_only=False)
cfg = bundle["config"]
tr_scores = []
for state in bundle["state_dicts"]:
    m = VisitTransformer(len(VOCAB), BLOCK_SIZE, d=cfg["d"], heads=cfg["heads"], blocks=cfg["blocks"], dropout=cfg["dropout"])
    m.load_state_dict(state)
    tr_scores.append(predict_proba(m, loader))
transformer = np.mean(tr_scores, axis=0)

counts_loader = make_loader(word_counts(ids), y, batch_size=4096)
lr_scores = []
for seed in (42, 43, 44):
    ck = torch.load(TUNED / f"logistic_regression_aa1b0357_s{seed}.pt", map_location="cpu", weights_only=False)
    m = LogisticRegression(VOCAB_SIZE - 1)
    m.load_state_dict(ck["state_dict"])
    lr_scores.append(predict_proba(m, counts_loader))
logistic = np.mean(lr_scores, axis=0)

scores = {"Random": np.random.default_rng(SEED).random(len(visits)),
          "Industry rule": industry_rule_score(visits),
          "Logistic regression": logistic,
          "Transformer": transformer}
print({k: (round(float(v.mean()), 4)) for k, v in scores.items()}, "| model config:", cfg)

{'Random': 0.5003, 'Industry rule': 0.2326, 'Logistic regression': 0.1329, 'Transformer': 0.0734} | model config: {'blocks': 3, 'd': 32, 'dropout': 0.2, 'heads': 2, 'lr': 0.001}


## 3. Ranking quality, with 95% ranges
AUC = chance a random come-back shopper outranks a random non-buyer (0.5 = coin flip). PR-AUC: random equals the base rate. Top-10% = come-back rate among the 10% of visits with the highest score. Ranges come from resampling shoppers (200 draws). For the main shop (electronics, test split) see `results/evaluation_quality.csv`: Transformer AUC 0.729, rule 0.562, logistic regression 0.716.

In [4]:
V = visits.copy()
for k, s in scores.items():
    V["s_" + k] = s

def metrics(d):
    out = {}
    yy = d["label"].to_numpy()
    for k in scores:
        s = d["s_" + k].to_numpy()
        out[f"{k}|auc"] = roc_auc_score(yy, s)
        out[f"{k}|pr_auc"] = average_precision_score(yy, s)
        out[f"{k}|top10"] = top_share_rate(yy, s)
    out["diff_T_minus_rule|auc"] = out["Transformer|auc"] - out["Industry rule|auc"]
    out["diff_T_minus_LR|auc"] = out["Transformer|auc"] - out["Logistic regression|auc"]
    out["diff_T_minus_rule|pr_auc"] = out["Transformer|pr_auc"] - out["Industry rule|pr_auc"]
    return out

point = metrics(V)
ranges = bootstrap(V, metrics, n=BOOT_N)
rows = []
for k in scores:
    rows.append({"model": k, "auc": point[f"{k}|auc"], "auc_lo": ranges[f"{k}|auc"][0], "auc_hi": ranges[f"{k}|auc"][1],
                 "pr_auc": point[f"{k}|pr_auc"], "pr_auc_lo": ranges[f"{k}|pr_auc"][0], "pr_auc_hi": ranges[f"{k}|pr_auc"][1],
                 "top10_rate": point[f"{k}|top10"], "top10_lo": ranges[f"{k}|top10"][0], "top10_hi": ranges[f"{k}|top10"][1],
                 "base_rate": float(y.mean())})
quality_table = pd.DataFrame(rows).set_index("model")
display(quality_table.round(4))
diff_rows = []
for name in ["diff_T_minus_rule|auc", "diff_T_minus_LR|auc", "diff_T_minus_rule|pr_auc"]:
    diff_rows.append({"difference": name.replace("|", " "), "value": point[name], "lo": ranges[name][0], "hi": ranges[name][1]})
diff_table = pd.DataFrame(diff_rows).set_index("difference")
display(diff_table.round(4))

,auc,auc_lo,auc_hi,pr_auc,pr_auc_lo,pr_auc_hi,top10_rate,top10_lo,top10_hi,base_rate
model,,,,,,,,,,
Random,0.4970,0.4922,0.5024,0.0779,0.0748,0.0815,0.0762,0.0713,0.0814,0.0787
Industry rule,0.7161,0.7076,0.7226,0.1636,0.1578,0.1691,0.2133,0.2057,0.2199,0.0787
Logistic regression,0.8361,0.8323,0.8394,0.2539,0.2431,0.2630,0.2894,0.2763,0.2988,0.0787
Transformer,0.8412,0.8365,0.8457,0.2875,0.2727,0.3005,0.3122,0.2932,0.3266,0.0787


,value,lo,hi
difference,,,
diff_T_minus_rule auc,0.1251,0.1164,0.1346
diff_T_minus_LR auc,0.0051,0.0033,0.0071
diff_T_minus_rule pr_auc,0.1239,0.1118,0.1350


## 4. Money at the middle setting
Same rules as `nudge/evaluate.py`: cart abandoners only, one discount per shopper per 7 days, 10% off. **Industry rule** = every cart abandoner gets it. **Nudge (Transformer)** = discount where `low <= score < high` from `cutoffs(g)` for g = 0 (stingy), 0.5 (middle) and 1 (generous); the high cut-off comes from electronics validation percentiles, so if cosmetics scores sit lower the policy shifts too. `wasted_eur` = 10% of what shoppers who came back anyway really bought. Cosmetics are cheap, so the euro amounts are small; the *percentage* saved is the comparable number.

In [5]:
def policies_for(g):
    low, high = cutoffs(g)
    return {"Industry rule (everyone)": lambda d: np.ones(len(d), dtype=bool),
            f"Nudge: Transformer, g={g}": lambda d: (d["s_Transformer"].to_numpy() >= low) & (d["s_Transformer"].to_numpy() < high)}

GS = [0.0, 0.5, 1.0]
allpol = {}
for g in GS:
    allpol.update(policies_for(g))
cut = {g: cutoffs(g) for g in GS}
print("cutoffs (low, high):", {g: tuple(round(x, 4) for x in c) for g, c in cut.items()})

def money_all(d):
    out = {}
    for name, rule in allpol.items():
        for k, v in money(d, rule(d)).items():
            out[f"{name}|{k}"] = v
    return out

mpoint = money_all(V)
mranges = bootstrap(V, money_all, n=BOOT_N)
mrows = []
for name in allpol:
    row = {"policy": name}
    for k in ["cart_abandoners", "discounts_sent", "wasted_eur", "came_back_anyway_with_discount", "spared_discount_came_back", "left_without_nudge"]:
        row[k] = mpoint[f"{name}|{k}"]
    row["wasted_eur_lo"], row["wasted_eur_hi"] = mranges[f"{name}|wasted_eur"]
    row["left_without_nudge_lo"], row["left_without_nudge_hi"] = mranges[f"{name}|left_without_nudge"]
    mrows.append(row)
money_table = pd.DataFrame(mrows).set_index("policy")
rule_w = money_table.loc["Industry rule (everyone)", "wasted_eur"]
money_table["wasted_vs_rule"] = money_table["wasted_eur"] / rule_w - 1
money_table["share_of_cart_abandoners_discounted"] = money_table["discounts_sent"] / money_table["cart_abandoners"]
display(money_table.round(3))


cutoffs (low, high): {0.0: (0.08, 0.1745), 0.5: (0.055, 0.2146), 1.0: (0.03, 0.2662)}


,cart_abandoners,discounts_sent,wasted_eur,came_back_anyway_with_discount,spared_discount_came_back,left_without_nudge,wasted_eur_lo,wasted_eur_hi,left_without_nudge_lo,left_without_nudge_hi,wasted_vs_rule,share_of_cart_abandoners_discounted
policy,,,,,,,,,,,,
Industry rule (everyone),42554,28941,21264.781,4415,4662,8951,20277.561,22357.821,8596.575,9343.025,0.000,0.680
"Nudge: Transformer, g=0.0",42554,14965,10037.250,2233,6844,20745,9501.954,10616.446,20285.400,21241.875,-0.528,0.352
"Nudge: Transformer, g=0.5",42554,21422,15005.037,3270,5807,15325,14285.739,15700.132,14950.475,15730.225,-0.294,0.503
"Nudge: Transformer, g=1.0",42554,26037,19004.629,4002,5075,11442,18148.828,19912.801,11089.700,11836.225,-0.106,0.612


## 5. Calibration
By score decile (equal-size groups): mean predicted chance vs the real come-back rate. If the ranking holds but the base rate differs between shops, the two columns drift apart even with a good AUC. On electronics the scores were well calibrated (a score of 0.2 meant about 20%).

In [6]:
def calibration(score, label, q=10):
    d = pd.DataFrame({"score": score, "label": label})
    d["decile"] = pd.qcut(d["score"].rank(method="first"), q, labels=False) + 1
    t = d.groupby("decile").agg(visits=("label", "size"), mean_predicted=("score", "mean"), actual_rate=("label", "mean"))
    t["ratio_actual_to_predicted"] = t["actual_rate"] / t["mean_predicted"]
    return t

cal = calibration(scores["Transformer"], y)
display(cal.round(4))
print(f"overall: mean predicted {scores['Transformer'].mean():.4f} vs actual {y.mean():.4f}  (ratio {y.mean() / scores['Transformer'].mean():.2f})")
print(f"by decile: Spearman between mean_predicted and actual_rate = {cal['mean_predicted'].corr(cal['actual_rate'], method='spearman'):.3f}")

,visits,mean_predicted,actual_rate,ratio_actual_to_predicted
decile,,,,
1,18298,0.0151,0.0093,0.6177
2,18298,0.0152,0.0055,0.3603
3,18298,0.0152,0.0039,0.2558
4,18298,0.0180,0.0094,0.5208
5,18298,0.0315,0.0195,0.6195
6,18298,0.0424,0.0367,0.8658
7,18298,0.0703,0.0765,1.0880
8,18298,0.1076,0.1154,1.0727
9,18298,0.1628,0.2014,1.2372


overall: mean predicted 0.0734 vs actual 0.0787  (ratio 1.07)
by decile: Spearman between mean_predicted and actual_rate = 0.960


## 6. Save results

In [7]:
quality_table.round(5).to_csv(OUT / "second_shop_quality.csv")
money_table.round(4).to_csv(OUT / "second_shop_money.csv")
cal.round(5).to_csv(OUT / "second_shop_calibration.csv")
summary = {"shop": "REES46 cosmetics (Oct-Nov 2019, 1 in 4 shoppers, no retraining)",
           "events_by_type_sampled": sampled, "remove_from_cart_dropped": int(dropped),
           "visits": int(len(visits)), "shoppers": int(stats["shoppers"]), "base_rate": float(y.mean()),
           "cart_abandoner_share": float((visits.cart_value > 0).mean()),
           "returning_share": float((visits.tokens.str[0] == "returning").mean()),
           "price_band_share_cosmetics": stats["price_band_share_cosmetics"], "price_band_share_main": stats["price_band_share_main"],
           "auc": {k: [float(point[f"{k}|auc"]), *map(float, ranges[f"{k}|auc"])] for k in scores},
           "pr_auc": {k: [float(point[f"{k}|pr_auc"]), *map(float, ranges[f"{k}|pr_auc"])] for k in scores},
           "top10": {k: [float(point[f"{k}|top10"]), *map(float, ranges[f"{k}|top10"])] for k in scores},
           "diff_auc_transformer_minus_rule": [float(point["diff_T_minus_rule|auc"]), *map(float, ranges["diff_T_minus_rule|auc"])],
           "mean_predicted": float(scores["Transformer"].mean()),
           "cutoffs": {str(g): list(c) for g, c in cut.items()}, "money": json.loads(money_table.round(4).to_json(orient="index"))}
(OUT / "second_shop_summary.json").write_text(json.dumps(summary, indent=2))
print("saved")

saved
